In [1]:
# !pip install -U pip wheel
# !pip install -q -r /Users/ryan/github/lltk/requirements.txt
import sys
sys.path.insert(0,'..')
sys.path.insert(0,'/Users/ryan/github/prosodic')
sys.path.insert(0,'/Users/ryan/github/lltk')
sys.path.insert(0,'/Users/ryan/github/logmap')
from llmdh import *
import prosodic
from prosodic import Word
prosodic.USE_CACHE=False
import plotnine as p9
p9.options.figure_size=11,8

In [2]:
def add_new_data(df):
    newdata = []
    # with open('/Users/ryan/Dropbox/Prof/Data/data.newpoems.json') as f: 
    with open('/Users/ryan/Downloads/data.newpoems2.json') as f: 
        for d in json.load(f):
            newdata.append({
                'model':d['prompt']['model'],
                'temperature':d['prompt']['temperature'],
                'prompt':d['prompt']['user_prompt'],
                'poem':d['response'].split('</think>')[-1].strip()
            })
    
    df2=pd.DataFrame(newdata)
    return pd.concat([df,df2])

def get_df_poems(min_lines=8):
    df_poems = pd.read_pickle('data.allpoems.pkl').fillna('')
    
    excl_prompts=[
        'Write an unryhmed poem in the style of Shakespeare\'s dramatic monologues.',
        'Write a poem in the style of Shakespeare\'s dramatic monologues.',
        'Write a poem in the style of e.e. cummings',
        # 'Write a poem in the style of Walt Whitman.',
        'Write a poem in the style of Wallace Stevens.',
        'Continue the following poem:\n\nTyping, typing, fingers on the keyboard\nThe keys crack and bend under sweat and weight,\n'
    ]

    df_poems = df_poems[~df_poems.prompt.isin(excl_prompts)]
    df_poems['num_lines'] = pd.to_numeric(df_poems['num_lines'], errors='coerce')
    odf = df_poems.query(f'num_lines >= {min_lines}')
    return add_new_data(odf)

In [3]:
pd.options.display.max_rows=100
df_poems = get_df_poems()
df_poems.drop_duplicates('poem').model.value_counts().sort_values()

model
ollama/deepseek-r1:8b                      573
ollama/llama2-uncensored:latest            575
ollama/darkmoon/olmo:7B-instruct-q6-k      589
llama2-uncensored:7b                       711
deepseek/deepseek-chat                     714
gemini-pro                                 974
claude-3-opus-20240229                     981
claude-3-sonnet-20240229                  1047
claude-3-haiku-20240307                   1101
gpt-4-turbo                               1167
gpt-3.5-turbo                             1916
ollama/llama3.1:70b                       2199
ollama/olmo2                              2228
ollama/llama3.1:8b                        2257
ollama/olmo2:13b                          2289
b. 1950-2000                              2479
b. 1850-1900                              6314
b. 1600-1650                              6787
b. 1650-1700                              7581
b. 1900-1950                              7856
b. 1800-1850                              8089
b. 1700

In [4]:
def get_txt_rhyming_data(txt, max_dist=0):
    bad_openings = ['Here is', 'Here\'s a', '**']
    lines1 = txt.split('\n')
    lines = [x for x in lines1 if not any(x.startswith(y) for y in bad_openings)][:1000]
    txt = '\n'.join(lines).strip()
    txt = '\n\n'.join([st for st in txt.split('\n\n') if st.count('\n')])
    poem = prosodic.Text(txt=txt)
    data = {'poem':txt}
    data['num_lines'] = poem.num_lines
    if data['num_lines']:
        rhymes = poem.get_rhyming_lines(max_dist=max_dist)
        rhymeset = set(rhymes.keys()) | set(rhymes.values())
        data['num_rhyming_lines'] = len(rhymeset)
        assert data['num_rhyming_lines'] <= data['num_lines']
    else:
        data['num_rhyming_lines'] = np.nan
    return data

In [5]:
poemtxt = df_poems.sample(n=1).iloc[0].poem
poem = prosodic.Text(poemtxt)

poem.get_rhyming_lines(max_dist=.5)

{Line(num=3, txt="\nYour grandeur charms, your beauty's lure in vain"): (0,
  Line(num=1, txt='Roll on, proud river, towards the mighty main,')),
 Line(num=4, txt="\n    The traveller's eye from yonder ancient pile."): (0,
  Line(num=2, txt="\n    And glow, gay shores, with summer's fostering smile,")),
 Line(num=7, txt='\nThe earliest temple reared by christian hands'): (0,
  Line(num=5, txt='\n\nFor there in solitary state it stands,')),
 Line(num=8, txt="\n    To teach a heathen world Jehovah's name."): (0,
  Line(num=6, txt='\n    While sheltering boughs involve its time-worn frame,')),
 Line(num=11, txt='\nEre from the wildering waste of waters dark'): (0,
  Line(num=9, txt='\n\nThus gleamed the altar, where the lonely ark')),
 Line(num=12, txt='\n    The rescued planet raised its mournful breast.'): (0,
  Line(num=10, txt="\n    Found for the patriarch's foot a place of rest,")),
 Line(num=20, txt="\n    The far cathedral, once their childhood's pride. --"): (0,
  Line(num=18, tx

In [6]:
print(poemtxt)

Roll on, proud river, towards the mighty main,
    And glow, gay shores, with summer's fostering smile,
Your grandeur charms, your beauty's lure in vain
    The traveller's eye from yonder ancient pile.

For there in solitary state it stands,
    While sheltering boughs involve its time-worn frame,
The earliest temple reared by christian hands
    To teach a heathen world Jehovah's name.

Thus gleamed the altar, where the lonely ark
    Found for the patriarch's foot a place of rest,
Ere from the wildering waste of waters dark
    The rescued planet raised its mournful breast.

Hail hallowed dome! whence first was herd to flow
    That strain of praise which heavenly choirs repeat,
While the stern savage stayed his quivering bow
    From echo's voice to woo that cadence sweet. -- 

Here, her young babe, the pensive matron brought,
    Here, the glad lover led his youthful bride,
And in thy solemn ordinance forgotten
    The far cathedral, once their childhood's pride. -- 

Were languag

In [7]:
# poem_txt = df_poems.query('model == "b. 1950-2000"').sample(n=1).iloc[0].poem
# print(poem_txt)
# poem = prosodic.Text(poem_txt)
# pprint(poem.get_rhyming_lines(max_dist=0))
# get_txt_rhyming_data(poem_txt)

In [8]:
def get_rhyme_data(fn='data.allpoems.pkl', force=False, min_lines=10, lim=None):
    df = get_df_poems(min_lines=min_lines)
    df['poem_hash'] = df['poem'].apply(hashstr)
    df = df.drop_duplicates('poem_hash')
    df = df.groupby(['model','prompt']).sample(n=100,replace=True)
    df = df.drop_duplicates('poem_hash')

    ofn = os.path.splitext(fn)[0]+'.rhyme_data4.tsv'
    done = set()
    if os.path.exists(ofn):
        with open(ofn) as f:
            for ln in f:
                id = ln.strip().split('\t')[0]
                if id:
                    done.add(id)
    print('Done already:',len(done))

    df = df[~df.poem_hash.isin(done)]
    poems = df.sample(frac=1).poem.drop_duplicates()

    col = ['poem', 'num_lines', 'num_rhyming_lines']
    file_exists = os.path.exists(ofn)
    with open(ofn,'a+') as of:
        if not file_exists:
            of.write('\t'.join(col)+'\n')
        for poem in tqdm(poems[:lim]):
            try:
                data = get_txt_rhyming_data(poem)
            except Exception as e:
                continue
            data['poem'] = hashstr(poem)
            outstr = '\t'.join(str(data[k]) for k in col) + '\n'
            of.write(outstr)
    return ofn

In [9]:
get_rhyme_data()

Done already: 33975


100%|██████████| 1631/1631 [02:19<00:00, 11.73it/s]


'data.allpoems.rhyme_data4.tsv'